In [2]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
print(device)
a = np.ones(5)
print(a)

b = torch.from_numpy(a)
print(b)

a += 1

print(a)
print(b)
b.div_(2)
c = b.div(2)
print(c)
c = torch.rand(5, device=device)
print(b.device)
print(c.device)

mps
[1. 1. 1. 1. 1.]
tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
[2. 2. 2. 2. 2.]
tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
tensor([0.5000, 0.5000, 0.5000, 0.5000, 0.5000], dtype=torch.float64)
cpu
mps:0


In [3]:
weights = torch.ones(4, requires_grad=True, device=device)

for epoch in range(2):
    model_output = (weights*3).sum()
    model_output.backward()
    print(weights.grad)
    weights.grad.zero_()

tensor([3., 3., 3., 3.], device='mps:0')
tensor([3., 3., 3., 3.], device='mps:0')


In [4]:
x = torch.tensor(1.0, device=device)
y = torch.tensor(2.0, device=device)

w = torch.tensor(1.0, device=device, requires_grad=True)

# forward pass
y_hat = w*x
loss = (y_hat - y).pow(2).sum()

print(loss)

# backward pass
loss.backward()
print(w.grad)

tensor(1., device='mps:0', grad_fn=<SumBackward0>)
tensor(-2., device='mps:0')


In [6]:
# f = w * x

# f = 2 * w

X = np.array([1, 2, 3, 4], dtype=np.float32)
Y = np.array([2, 4, 6, 8], dtype=np.float32)

w = 0.0

def forward(x):
    return w*x

def loss(y, y_hat):
    return ((y_hat - y)**2).mean()

# MSE = 1/N * (w*x - y)**2
# dl/dw = (2*x)/N * (w*x - y)

def gradient(x, y, y_hat):
    return np.dot(2*x, y_hat-y).mean()

print(f'Prediction before training: f(5) = {forward(5):.3f}')

learning_rate = 0.01
n_iters = 100

for epoch in range(n_iters):
    y_hat = forward(X)
    l = loss(Y, y_hat)
    dw = gradient(X, Y, y_hat)

    w -= learning_rate * dw

    if epoch % 10 == 0:
        print(f'epoch {epoch+1}: w = {w:.3f}, loss = {l:.8f}')


print(f'Prediction after training: f(5) = {forward(5):.3f}')

Prediction before training: f(5) = 0.000
epoch 1: w = 1.200, loss = 30.00000000
epoch 11: w = 2.000, loss = 0.00000033
epoch 21: w = 2.000, loss = 0.00000000
epoch 31: w = 2.000, loss = 0.00000000
epoch 41: w = 2.000, loss = 0.00000000
epoch 51: w = 2.000, loss = 0.00000000
epoch 61: w = 2.000, loss = 0.00000000
epoch 71: w = 2.000, loss = 0.00000000
epoch 81: w = 2.000, loss = 0.00000000
epoch 91: w = 2.000, loss = 0.00000000
Prediction after training: f(5) = 10.000


In [9]:
device = "cpu"
X = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
Y = torch.tensor([2, 4, 6, 8], dtype=torch.float32)

w = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)


def forward(x):
    return w * x

def loss(y, y_hat):
    return ((y_hat - y) ** 2).mean()


print(f'Prediction before training: f(5) = {forward(5):.3f}')

learning_rate = 0.01
n_iters = 100

for epoch in range(n_iters):
    y_hat = forward(X)
    l = loss(Y, y_hat)
    l.backward() # dl/dw


    with torch.no_grad():
        w -= learning_rate * w.grad

    w.grad.zero_()

    if epoch % 10 == 0:
        print(f'epoch {epoch + 1}: w = {w:.3f}, loss = {l:.8f}')

print(f'Prediction after training: f(5) = {forward(5):.3f}')

Prediction before training: f(5) = 0.000
epoch 1: w = 0.300, loss = 30.00000000
epoch 11: w = 1.665, loss = 1.16278565
epoch 21: w = 1.934, loss = 0.04506890
epoch 31: w = 1.987, loss = 0.00174685
epoch 41: w = 1.997, loss = 0.00006770
epoch 51: w = 1.999, loss = 0.00000262
epoch 61: w = 2.000, loss = 0.00000010
epoch 71: w = 2.000, loss = 0.00000000
epoch 81: w = 2.000, loss = 0.00000000
epoch 91: w = 2.000, loss = 0.00000000
Prediction after training: f(5) = 10.000


In [14]:
import torch.nn as nn

X = torch.tensor([[1], [2], [3], [4]], dtype=torch.float32)
Y = torch.tensor([[2], [4], [6], [8]], dtype=torch.float32)

X_test = torch.tensor([5], dtype=torch.float32)

n_samples, n_features = X.shape

input_size = n_features
output_size = n_features
#model = nn.Linear(in_features=input_size, out_features=output_size)

class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearRegression, self).__init__()
        self.lin = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.lin(x)

model = LinearRegression(input_size, output_size)

print(f'Prediction before training: f(5) = {model(X_test).item():.3f}')

learning_rate = 0.05
n_iters = 100

loss = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(n_iters):
    y_hat = model(X)
    l = loss(Y, y_hat)
    l.backward()  # dl/dw

    optimizer.step()

    optimizer.zero_grad()

    if epoch % 10 == 0:
        [w, b] = model.parameters()
        print(f'epoch {epoch + 1}: w = {w[0][0].item():.3f}, loss = {l:.8f}')

print(f'Prediction after training: f(5) = {model(X_test).item():.3f}')

Prediction before training: f(5) = -4.178
epoch 1: w = 1.210, loss = 58.79521942
epoch 11: w = 1.677, loss = 0.15555060
epoch 21: w = 1.722, loss = 0.11504431
epoch 31: w = 1.761, loss = 0.08508608
epoch 41: w = 1.794, loss = 0.06292917
epoch 51: w = 1.823, loss = 0.04654205
epoch 61: w = 1.848, loss = 0.03442221
epoch 71: w = 1.869, loss = 0.02545846
epoch 81: w = 1.888, loss = 0.01882891
epoch 91: w = 1.903, loss = 0.01392576
Prediction after training: f(5) = 9.826
